# Reproducibility Audit Notebook

This notebook audits a reconstructed validation package. It preserves author-provided values, recomputes only what can be recomputed from recovered inputs, and labels missing-data limitations explicitly.

## Scope

- This is not full raw reproduction of 45 experimental sessions.
- Session-level rows, rubric mappings, correlation source matrices, and the exact TDI formula/normalization are not fully recovered.
- No values are fabricated or forced to match.

In [ ]:
from pathlib import Path
import sys

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import PathConfig
from src.export_paper_assets import export_assets
from src.run_pipeline import run
from src.utils.io import load_json

paths = PathConfig(ROOT)
source = load_json(paths.provided_paper_data)
run(ROOT)
export_assets(ROOT)

## Reproducibility Overview

In [ ]:
overview = pd.read_csv(paths.reproducibility_overview)
validation = pd.read_csv(paths.validation_report_grouped)
paper_ready = pd.read_csv(paths.paper_ready_metrics)
display(overview)
display(validation['category'].value_counts().rename_axis('category').reset_index(name='count'))

## Recomputed Summaries

In [ ]:
cell_summary = pd.read_csv(paths.per_cell_summary)
approach_summary = pd.read_csv(paths.per_approach_summary)
display(cell_summary.head(10))
display(approach_summary)

## Preserved Paper Values

In [ ]:
display(paper_ready.head(30))
display(paper_ready['reproducibility_status'].value_counts().rename_axis('status').reset_index(name='count'))

## Validation Rows Needing Reviewer Attention

In [ ]:
attention = validation[validation['category'] != 'MATCH']
display(attention[['metric_name', 'level', 'expected_value', 'computed_value', 'category', 'reproducibility_status', 'explanation']])

## Paper Figures

The following figures are generated by `src.export_paper_assets` using preserved source values where full recomputation is unavailable.

In [ ]:
figure_paths = [
    paths.reports_figures / 'figure1_development_time.png',
    paths.reports_figures / 'figure2_maintainability_index.png',
    paths.reports_figures / 'figure3_pre_release_bugs.png',
    paths.reports_figures / 'figure4_security_issues.png',
    paths.reports_figures / 'figure5_rework_ratio.png',
    paths.reports_figures / 'figure6_layer_quality.png',
]
for figure_path in figure_paths:
    img = plt.imread(figure_path)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(figure_path.name)
    plt.show()

## Final Audit Summary

In [ ]:
true_mismatch = int(overview.loc[overview['category'] == 'TRUE_MISMATCH', 'count'].iloc[0])
print('This repository is a reconstructed validation package, not full raw reproduction.')
print(f'TRUE_MISMATCH count: {true_mismatch}')
print('Critical contradictions:', 'none' if true_mismatch == 0 else true_mismatch)
print('Primary limitations: missing 45 session rows, missing rubric mappings, unrecovered TDI formula, preserved correlations.')